# 🧠 GYAN Analytics — Training Data Generator
**Team:** MPR_CS_26_T7 | Gagan G, Esha Kumara, Bharathkumar G, Atharva Joshi

### Domains Covered:
🎓 Education | 🏥 Healthcare | 💼 Business | 💰 Finance | 🏛️ Government
👥 HR | 🌾 Agriculture | 🚗 Transport | ⚽ Sports | 🕌 **Religious Donations** ⭐

**Just run each cell top to bottom. No coding needed.**

## 📦 Step 1 — Install Libraries

In [ ]:
!pip install -q pandas numpy scikit-learn scipy tqdm
print('✅ Libraries installed')

## 📂 Step 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os

# ✏️ CHANGE THIS if your folder is named differently
DATASETS_FOLDER = '/content/drive/MyDrive/GYAN_TRAINING_DATA'
OUTPUT_FOLDER   = '/content/drive/MyDrive/GYAN_TRAINING_DATA/output'
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

csv_files = [f for f in os.listdir(DATASETS_FOLDER) if f.endswith('.csv')]
print(f'✅ Found {len(csv_files)} CSV datasets:')
for f in csv_files:
    size = os.path.getsize(os.path.join(DATASETS_FOLDER, f)) / 1024
    print(f'   📄 {f} ({size:.1f} KB)')

## 🔬 Step 3 — Analysis Engine

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from scipy import stats
import json, math, warnings
warnings.filterwarnings('ignore')

def sanitize(obj):
    if isinstance(obj, float) and (math.isnan(obj) or math.isinf(obj)): return None
    if isinstance(obj, dict):  return {k: sanitize(v) for k, v in obj.items()}
    if isinstance(obj, list):  return [sanitize(v) for v in obj]
    if isinstance(obj, np.integer):  return int(obj)
    if isinstance(obj, np.floating): return float(obj)
    return obj

def analyse_dataframe(df, filename='dataset'):
    findings = {}
    df = df.copy()
    findings['rows'] = len(df)
    findings['columns'] = len(df.columns)
    findings['column_names'] = list(df.columns[:20])
    findings['missing_pct'] = round(df.isnull().mean().mean() * 100, 2)
    findings['duplicate_pct'] = round(df.duplicated().mean() * 100, 2)

    num_df = df.select_dtypes(include=[np.number])
    num_cols = num_df.columns.tolist()
    findings['numeric_columns'] = num_cols

    col_stats = {}
    for col in num_cols[:15]:
        s = num_df[col].dropna()
        if len(s) < 2: continue
        col_stats[col] = {
            'mean': round(float(s.mean()), 3), 'median': round(float(s.median()), 3),
            'std':  round(float(s.std()), 3),  'min': round(float(s.min()), 3),
            'max':  round(float(s.max()), 3),  'missing': int(num_df[col].isnull().sum()),
            'skew': round(float(s.skew()), 3),
        }
    findings['column_stats'] = col_stats

    date_cols = [c for c in df.columns if any(k in c.lower() for k in ['date','year','month','time','week','day','period','occasion','festival'])]
    findings['has_time_series'] = len(date_cols) > 0
    findings['time_columns'] = date_cols

    trends = {}
    if date_cols and num_cols:
        try:
            time_col = date_cols[0]
            df_s = df.copy()
            df_s[time_col] = pd.to_datetime(df_s[time_col], errors='coerce')
            df_s = df_s.dropna(subset=[time_col]).sort_values(time_col)
            for col in num_cols[:5]:
                s = df_s[col].dropna()
                if len(s) > 5:
                    slope, _, r, _, _ = stats.linregress(np.arange(len(s)), s)
                    trends[col] = {
                        'slope': round(float(slope), 4),
                        'direction': 'increasing' if slope > 0.01 else ('decreasing' if slope < -0.01 else 'stable'),
                        'r_squared': round(float(r**2), 3)
                    }
        except: pass
    findings['trends'] = trends

    if len(num_cols) >= 1 and len(num_df.dropna()) >= 5:
        try:
            clean = num_df.fillna(num_df.median())
            valid = [c for c in num_cols if clean[c].std() > 0]
            if valid:
                X = StandardScaler().fit_transform(clean[valid])
                preds = IsolationForest(contamination=0.05, random_state=42, n_estimators=100).fit_predict(X)
                ac = int((preds == -1).sum())
                findings['anomalies'] = {'count': ac, 'percentage': round(ac/len(df)*100, 2), 'evaluated_cols': valid[:8]}
        except: findings['anomalies'] = {'count': 0, 'percentage': 0, 'evaluated_cols': []}

        try:
            if len(num_cols) >= 2 and len(num_df.dropna()) >= 10:
                clean = num_df.fillna(num_df.median())
                k = min(4, max(2, len(df) // 50))
                labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(StandardScaler().fit_transform(clean))
                findings['clusters'] = {'k': k, 'segments': [{'id': i+1, 'size': int((labels==i).sum()), 'percentage': round((labels==i).sum()/len(df)*100, 1)} for i in range(k)]}
        except: findings['clusters'] = {'k': 0, 'segments': []}

        try:
            if len(num_cols) >= 2:
                corr = num_df.corr()
                pairs = []
                for i in range(len(num_cols)):
                    for j in range(i+1, len(num_cols)):
                        v = corr.loc[num_cols[i], num_cols[j]]
                        if not np.isnan(v) and abs(v) > 0.3:
                            pairs.append({'col_a': num_cols[i], 'col_b': num_cols[j], 'r': round(float(v), 3), 'strength': 'strong' if abs(v)>=0.7 else 'moderate'})
                findings['correlations'] = sorted(pairs, key=lambda x: abs(x['r']), reverse=True)[:10]
        except: findings['correlations'] = []

    return sanitize(findings)

print('✅ Analysis engine ready')

## ✍️ Step 4 — Domain Detector + Insight Engine

In [ ]:
import random

# ============================================================
# DOMAIN KEYWORDS — used to auto-detect domain from column names
# ============================================================
DOMAIN_KEYWORDS = {
    'education':   ['student','mark','grade','score','attendance','subject','exam','pass','fail','gpa'],
    'healthcare':  ['patient','hospital','disease','blood','pressure','age','diagnosis','treatment','death','survival'],
    'business':    ['sales','revenue','profit','customer','product','order','price','discount','region','store'],
    'finance':     ['transaction','fraud','amount','balance','credit','loan','payment','bank','stock','return'],
    'government':  ['population','census','crime','budget','state','district','election','employment','poverty'],
    'hr':          ['employee','salary','attrition','department','job','performance','hire','resign','tenure'],
    'agriculture': ['crop','yield','rainfall','fertilizer','soil','farm','harvest','irrigation','seed','pesticide'],
    'transport':   ['trip','fare','distance','driver','route','flight','delay','traffic','speed','vehicle'],
    'sports':      ['player','match','score','team','goal','wicket','run','point','win','league'],
    # ⭐ NEW DOMAIN
    'religious':   ['donation','temple','mosque','church','gurudwara','hundi','offering','festival','prayer','charity','tithe','zakat','daan','prasad'],
}

def detect_domain(df):
    cols_lower = ' '.join(df.columns).lower()
    scores = {d: sum(1 for kw in kws if kw in cols_lower) for d, kws in DOMAIN_KEYWORDS.items()}
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else 'general'


def findings_to_input_text(findings, domain):
    parts = [
        f"Domain: {domain}",
        f"Rows: {findings['rows']} | Columns: {findings['columns']}",
        f"Missing data: {findings['missing_pct']}% | Duplicates: {findings['duplicate_pct']}%",
    ]
    if findings.get('column_stats'):
        parts.append("Column stats: " + " | ".join(
            f"{col}(mean={s['mean']},std={s['std']},min={s['min']},max={s['max']})"
            for col, s in list(findings['column_stats'].items())[:6]
        ))
    if findings.get('anomalies'):
        a = findings['anomalies']
        parts.append(f"Anomalies: {a['count']} rows flagged ({a['percentage']}% of data)")
    if findings.get('clusters', {}).get('k', 0) > 0:
        segs = findings['clusters']['segments']
        parts.append(f"Clusters (k={findings['clusters']['k']}): " + " | ".join(f"Segment {s['id']}: {s['size']} rows ({s['percentage']}%)" for s in segs))
    if findings.get('correlations'):
        parts.append("Key correlations: " + " | ".join(f"{c['col_a']} ↔ {c['col_b']}: r={c['r']} ({c['strength']})" for c in findings['correlations'][:5]))
    if findings.get('trends'):
        parts.append("Trends: " + " | ".join(f"{col}: {t['direction']} (R²={t['r_squared']})" for col, t in list(findings['trends'].items())[:4]))
    return " || ".join(parts)


def findings_to_insight(findings, domain):
    parts = []

    # Overview
    parts.append(f"This {domain} dataset contains {findings['rows']:,} records across {findings['columns']} variables.")

    # Data quality
    mp = findings['missing_pct']
    if mp == 0:    parts.append("Data quality is excellent with no missing values detected.")
    elif mp < 5:   parts.append(f"Data quality is good with only {mp}% missing values — minor imputation recommended.")
    elif mp < 20:  parts.append(f"Moderate data quality issue: {mp}% missing values detected. Imputation or column removal advised.")
    else:          parts.append(f"Significant data quality problem: {mp}% missing values. Data collection process needs review.")

    # Domain-specific context
    domain_context = {
        'education':   'This data reflects student academic performance and engagement metrics.',
        'healthcare':  'This data relates to patient health indicators and clinical outcomes.',
        'business':    'This data captures commercial performance and customer behavior.',
        'finance':     'This data covers financial transactions and monetary risk indicators.',
        'government':  'This data represents public sector metrics and civic indicators.',
        'hr':          'This data reflects workforce dynamics and employee lifecycle metrics.',
        'agriculture': 'This data covers agricultural output, environmental conditions, and farming inputs.',
        'transport':   'This data captures mobility patterns, logistics, and travel metrics.',
        'sports':      'This data reflects athletic performance statistics and competition outcomes.',
        'religious':   'This data captures donation patterns, collection records, and giving trends across religious events and occasions.',
    }
    if domain in domain_context:
        parts.append(domain_context[domain])

    # Skew
    if findings.get('column_stats'):
        high_skew = [(c, s['skew']) for c, s in findings['column_stats'].items() if s.get('skew') and abs(s['skew']) > 1.5]
        if high_skew:
            parts.append(f"Skewed distributions detected in {', '.join(c for c,_ in high_skew[:3])} — suggesting outlier influence or non-normal patterns worth investigating.")

    # Anomalies
    if findings.get('anomalies'):
        a = findings['anomalies']
        if a['count'] == 0:
            parts.append("No significant anomalies detected — data follows consistent patterns.")
        elif a['percentage'] < 5:
            parts.append(f"{a['count']} anomalous records identified ({a['percentage']}% of data). These outliers deserve manual review as they deviate significantly from normal patterns.")
            if domain == 'religious':
                parts.append("These unusual donation amounts may represent exceptional events, large donors, or entries requiring audit verification.")
        else:
            parts.append(f"High anomaly rate detected: {a['count']} records ({a['percentage']}%) show unusual patterns. This level suggests either data quality issues or genuinely exceptional cases requiring investigation.")

    # Clusters
    if findings.get('clusters', {}).get('k', 0) > 0:
        k = findings['clusters']['k']
        segs = findings['clusters']['segments']
        largest = max(segs, key=lambda x: x['size'])
        smallest = min(segs, key=lambda x: x['size'])
        parts.append(f"Data naturally groups into {k} distinct segments. The dominant group contains {largest['percentage']}% of records, while the smallest segment ({smallest['percentage']}%) may represent a niche or at-risk subgroup.")
        if domain == 'religious':
            parts.append(f"These segments may correspond to different donor types: regular contributors, festival-driven donors, and occasional givers — each requiring a different engagement strategy.")

    # Correlations
    if findings.get('correlations'):
        top = findings['correlations'][0]
        direction = 'positively' if top['r'] > 0 else 'negatively'
        parts.append(f"Strongest relationship found between '{top['col_a']}' and '{top['col_b']}' (r={top['r']}): these variables are {direction} correlated at a {top['strength']} level — changes in one are likely to influence the other.")
        if domain == 'religious' and len(findings['correlations']) > 1:
            parts.append("This suggests that specific occasions or time periods are strong predictors of donation volume — useful for planning fundraising events.")

    # Trends
    if findings.get('trends'):
        increasing = [(c, t) for c, t in findings['trends'].items() if t['direction'] == 'increasing']
        decreasing = [(c, t) for c, t in findings['trends'].items() if t['direction'] == 'decreasing']
        if increasing:
            parts.append(f"Positive trends observed in {', '.join(c for c,_ in increasing[:2])} — consistent growth pattern detected over time.")
        if decreasing:
            parts.append(f"Declining trends in {', '.join(c for c,_ in decreasing[:2])} require attention — root cause analysis and corrective action recommended before the trend worsens.")
            if domain == 'religious':
                parts.append("Declining donation trends in religious institutions may reflect shifting demographics, reduced community engagement, or transition to digital payment modes not yet captured in records.")

    # Recommendations
    recs = []
    if findings.get('anomalies', {}).get('count', 0) > 0:
        recs.append(f"Investigate the {findings['anomalies']['count']} flagged anomalous records")
        if domain == 'religious': recs.append("Verify large donation entries against physical receipts or UPI transaction logs")
    if findings.get('missing_pct', 0) > 5:
        recs.append("Address missing data through imputation or data re-collection")
    if findings.get('correlations'):
        top = findings['correlations'][0]
        recs.append(f"Monitor '{top['col_a']}' as a key driver influencing '{top['col_b']}'")
    if findings.get('clusters', {}).get('k', 0) > 0:
        recs.append(f"Develop targeted strategies for each of the {findings['clusters']['k']} identified segments")
    if recs:
        parts.append("Recommended actions: " + "; ".join(f"({i+1}) {r}" for i, r in enumerate(recs)) + ".")

    return ' '.join(parts)

print('✅ Domain detector + insight engine ready')
print(f'   Domains supported: {", ".join(DOMAIN_KEYWORDS.keys())}')

## 🏭 Step 5 — Generate Training Examples From Kaggle CSVs

In [ ]:
from tqdm import tqdm

all_examples = []

for csv_file in tqdm(csv_files, desc='Processing datasets'):
    filepath = os.path.join(DATASETS_FOLDER, csv_file)
    try:
        df = None
        for enc in ['utf-8', 'latin1', 'cp1252']:
            try: df = pd.read_csv(filepath, encoding=enc, low_memory=False); break
            except: continue
        if df is None or df.empty or len(df) < 10:
            print(f'  ⚠️  Skipping {csv_file} — too few rows'); continue

        domain = detect_domain(df)
        print(f'  📄 {csv_file} → domain: {domain}, rows: {len(df):,}')

        n_samples = min(20, max(5, len(df) // 100))
        sample_sizes = [s for s in [50, 100, 200, 500, 1000, len(df)] if s <= len(df)]

        for sample_size in sample_sizes:
            for _ in range(n_samples):
                try:
                    sample_df = df.sample(n=min(sample_size, len(df)), random_state=random.randint(0, 9999))
                    findings = analyse_dataframe(sample_df, csv_file)
                    if not findings.get('numeric_columns'): continue
                    out = findings_to_insight(findings, domain)
                    if len(out) > 100:
                        all_examples.append({'input': findings_to_input_text(findings, domain), 'output': out, 'domain': domain, 'source': csv_file})
                except: continue

        print(f'     → {len(all_examples):,} total examples so far')
    except Exception as e:
        print(f'  ❌ Error on {csv_file}: {e}')

print(f'\n✅ Total training examples from Kaggle CSVs: {len(all_examples):,}')

## 🕌 Step 6 — Generate RELIGIOUS DONATIONS Synthetic Data (Special Domain)

In [ ]:
# ============================================================
# RELIGIOUS DONATIONS — Realistic synthetic data generator
# ============================================================
import random

INSTITUTIONS    = ['Temple', 'Mosque', 'Church', 'Gurudwara', 'Monastery', 'Dargah', 'Synagogue']
OCCASIONS       = ['Daily prayer', 'Weekly service', 'Festival', 'Special ceremony', 'Marriage blessing', 'Death rites', 'Charity drive', 'Annual celebration']
FESTIVALS       = ['Diwali', 'Navratri', 'Eid ul-Fitr', 'Eid ul-Adha', 'Christmas', 'Easter', 'Gurpurab', 'Holi', 'Dussehra', 'Pongal', 'Onam', 'Baisakhi']
COLLECTION_TYPES = ['Hundi/Donation box', 'Online/UPI', 'Cheque', 'Cash', 'Bank transfer', 'Prasad offering']
REGIONS         = ['North India', 'South India', 'East India', 'West India', 'Central India', 'Northeast India']

# Festival multipliers — realistic spike patterns
FESTIVAL_MULTIPLIER = {'Diwali': 4.2, 'Navratri': 3.8, 'Eid ul-Fitr': 3.5, 'Christmas': 3.0,
                       'Gurpurab': 2.9, 'Easter': 2.5, 'Holi': 2.3, 'Dussehra': 2.6,
                       'Pongal': 2.1, 'Onam': 2.0, 'Baisakhi': 2.2, 'Eid ul-Adha': 3.3}

def generate_religious_df(n_rows=None):
    n_rows = n_rows or random.randint(50, 500)
    institution = random.choice(INSTITUTIONS)
    base_donation = random.uniform(500, 15000)   # base daily/weekly average
    rows = []
    for _ in range(n_rows):
        is_festival = random.random() < 0.15
        festival = random.choice(FESTIVALS) if is_festival else None
        multiplier = FESTIVAL_MULTIPLIER.get(festival, 1.0) if festival else 1.0
        occasion = festival if festival else random.choice(OCCASIONS)
        month = random.randint(1, 12)
        day_of_week = random.randint(0, 6)  # 0=Mon, 4=Fri, 6=Sun

        # Friday/Sunday peak for mosques/churches
        day_multiplier = 1.8 if day_of_week in [4, 6] else 1.0

        amount = abs(random.gauss(base_donation * multiplier * day_multiplier, base_donation * 0.3))
        donor_count = max(1, int(amount / random.uniform(50, 500)))

        # Occasional large donor
        if random.random() < 0.03:
            amount *= random.uniform(5, 15)

        rows.append({
            'donation_amount': round(amount, 2),
            'donor_count': donor_count,
            'month': month,
            'day_of_week': day_of_week,
            'year': random.randint(2018, 2024),
            'is_festival': int(is_festival),
            'collection_type_code': COLLECTION_TYPES.index(random.choice(COLLECTION_TYPES)),
            'region_code': REGIONS.index(random.choice(REGIONS)),
        })
    return pd.DataFrame(rows)

RELIGIOUS_SYNTHETIC = 8000
religious_examples = []

print(f'🕌 Generating {RELIGIOUS_SYNTHETIC} religious donation examples...')
for i in tqdm(range(RELIGIOUS_SYNTHETIC)):
    try:
        df_r = generate_religious_df()
        findings = analyse_dataframe(df_r)
        if not findings.get('numeric_columns'): continue
        out = findings_to_insight(findings, 'religious')
        if len(out) > 100:
            religious_examples.append({
                'input': findings_to_input_text(findings, 'religious'),
                'output': out,
                'domain': 'religious',
                'source': 'synthetic_religious'
            })
    except: continue

all_examples.extend(religious_examples)
print(f'\n✅ Religious examples generated: {len(religious_examples):,}')
print(f'   Total examples so far: {len(all_examples):,}')

## 🎲 Step 7 — Generate SYNTHETIC Examples for All Other Domains

In [ ]:
DOMAIN_TEMPLATES = {
    'education':   {'cols': ['marks','attendance','grade','homework','exam_score'],            'mean_range': (30, 95),      'std_range': (5, 25)},
    'healthcare':  {'cols': ['blood_pressure','sugar_level','age','bmi','cholesterol'],        'mean_range': (60, 200),     'std_range': (10, 50)},
    'business':    {'cols': ['sales','revenue','profit','customers','orders'],                 'mean_range': (1000, 100000),'std_range': (500, 20000)},
    'finance':     {'cols': ['amount','balance','transactions','credit_score','loan'],         'mean_range': (5000, 500000),'std_range': (1000, 50000)},
    'government':  {'cols': ['population','budget','crime_rate','employment','gdp'],           'mean_range': (10000,1000000),'std_range': (5000,100000)},
    'hr':          {'cols': ['salary','experience','performance_score','age','tenure'],        'mean_range': (30000,150000),'std_range': (5000, 30000)},
    'agriculture': {'cols': ['yield_kg','rainfall_mm','temperature','fertilizer_kg','area'],  'mean_range': (100, 5000),   'std_range': (50, 1000)},
    'transport':   {'cols': ['fare','distance_km','trip_duration','speed_kmh','rating'],      'mean_range': (10, 5000),    'std_range': (5, 500)},
    'sports':      {'cols': ['score','runs','goals','assists','rating','speed'],               'mean_range': (10, 300),     'std_range': (5, 50)},
}

def generate_synthetic_df(domain, n_rows=None):
    t = DOMAIN_TEMPLATES[domain]
    n_rows = n_rows or random.randint(50, 2000)
    cols = random.sample(t['cols'], random.randint(3, len(t['cols'])))
    data = {}
    for col in cols:
        mean = random.uniform(*t['mean_range'])
        std  = random.uniform(*t['std_range'])
        vals = np.random.normal(mean, std, n_rows)
        if random.random() < 0.4:
            idx = np.random.choice(n_rows, random.randint(1, max(1, n_rows//20)), replace=False)
            vals[idx] *= random.uniform(3, 8)
        data[col] = vals
    return pd.DataFrame(data)

SYNTHETIC_TARGET = 35000
domains = list(DOMAIN_TEMPLATES.keys())

print(f'Generating {SYNTHETIC_TARGET} synthetic examples across {len(domains)} domains...')
for i in tqdm(range(SYNTHETIC_TARGET)):
    try:
        domain = random.choice(domains)
        df_syn = generate_synthetic_df(domain)
        findings = analyse_dataframe(df_syn)
        if not findings.get('numeric_columns'): continue
        out = findings_to_insight(findings, domain)
        if len(out) > 100:
            all_examples.append({'input': findings_to_input_text(findings, domain), 'output': out, 'domain': domain, 'source': 'synthetic'})
    except: continue

# Final stats
from collections import Counter
domain_counts  = Counter(e['domain'] for e in all_examples)
source_counts  = Counter('synthetic' if 'synthetic' in e['source'] else 'kaggle' for e in all_examples)

print(f'\n✅ TOTAL TRAINING EXAMPLES: {len(all_examples):,}')
print('\nDomain distribution:')
for d, c in domain_counts.most_common(): print(f'  {d:15s}: {c:,}')
print(f'\nKaggle real: {source_counts["kaggle"]:,} | Synthetic: {source_counts["synthetic"]:,}')

## 💾 Step 8 — Save to Google Drive

In [ ]:
import json

random.shuffle(all_examples)
split      = int(len(all_examples) * 0.9)
train_data = all_examples[:split]
val_data   = all_examples[split:]

train_path = os.path.join(OUTPUT_FOLDER, 'gyan_train.jsonl')
val_path   = os.path.join(OUTPUT_FOLDER, 'gyan_val.jsonl')

with open(train_path, 'w') as f:
    for ex in train_data: f.write(json.dumps(ex) + '\n')

with open(val_path, 'w') as f:
    for ex in val_data: f.write(json.dumps(ex) + '\n')

print('✅ Saved to Google Drive:')
print(f'   Training:   {len(train_data):,} examples → gyan_train.jsonl')
print(f'   Validation: {len(val_data):,} examples  → gyan_val.jsonl')
print(f'\n📂 Location: {OUTPUT_FOLDER}')
print('\n🎉 Done! Open GYAN_Model_Finetuning.ipynb to start training.')

## ✅ What's Next

Your files are saved:
- `gyan_train.jsonl` — training set (~90%)
- `gyan_val.jsonl` — validation set (~10%)

**Next step:** Open `GYAN_Model_Finetuning.ipynb`
- Loads these files
- Fine-tunes Qwen2.5-0.5B on free Colab GPU
- Takes 2-4 hours
- Saves trained model to Drive